# Getting Started

REPLICA lets you evaluate a comprehension model against human responses. Bring
an existing feature model or implement your own: a `FeatureModel` maps a stimulus
to per-unit outputs, and `ModelFeature` binds a source to a named extraction. Stored measurements
and stimulus properties are also features. `EvaluationSpec` assigns them to
targets, predictors and controls; `AnalysisConfig` chooses how to fit and evaluate their
relationship to the responses. A language model with surprisal is one example.

This guide loads a dataset, evaluates that example and inspects its results.
The [extension tutorial](extending_replica.ipynb) shows how to supply your own
model and features.

## Install and fetch example data

Use Python 3.11 or newer. This example also needs R and its `lme4` package
(`install.packages("lme4")` in R); install these before the Python dependencies.
From a fresh checkout:

```bash
git clone https://github.com/replicaverse/replica.git
cd replica
python -m venv .venv
source .venv/bin/activate
pip install -e ".[models,r]"
replica datasets
replica fetch provo
```

Run this notebook in the Python environment above. The example uses GPT-2,
which downloads on first use. See the [environment guide](../guides/environment.md#optional-capabilities) for
other optional dependencies.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

## 1. Load a dataset

Every dataset shares the same interface. Use `describe_dataset()` to inspect valid keys and options without loading data, then `load_dataset()` to construct and load it in one call. `load_dataset()` reads from `$PSYCHO_DATA_DIR`, or from `datasets/` in a source checkout when that variable is unset.

In [ ]:
from replica.dataset import describe_dataset, load_dataset

print(describe_dataset("provo"))
ds = load_dataset("provo")

print(f"Dataset:        {ds.info.name}")
print(f"Language:       {ds.info.language}")
print(f"Stimuli:        {len(ds.stimuli())}")
print(f"Participants:   {len(ds.participants)}")
print(f"Measures:       {ds.measures}")
print(f"Default DV:     {ds.default_measure}")

### Stimuli

A `Stimulus` contains text, an ID and annotation layers. PROVO's `"words"`
layer identifies the measured word spans; `unit_strings()` reads their text.

In [ ]:
stim = ds.stimuli()[0]
print(f"ID:     {stim.stimulus_id}")
print(f"Text:   {stim.text[:100]}...")
print(f"Words:  {len(stim.layers['words'])}")
print(f"Layers: {list(stim.layers.keys())}")
print()
# Words are accessible as strings
print("First 10 words:", stim.unit_strings()[:10])

### Responses

A `Datum` contains a response array for one stimulus and, where available, one
participant. Its values align with the spans in the named annotation layer.

In [ ]:
import numpy as np

# First-run duration restricted to first-pass entry
data = ds.data(measure="first_run_in_first_pass")
print(f"Total Datum objects: {len(data)}")

datum = data[0]
print("\nExample Datum:")
print(f"  Stimulus:    {datum.stimulus.stimulus_id}")
print(f"  Participant: {datum.participant_id}")
print(f"  Measure:     {datum.measure_name}")
print(f"  Values:      {datum.values[:10]}  (first 10 words)")
print(
    f"  Shape:       {datum.values.shape}  ({len(datum.stimulus.layers['words'])} words)"
)

# Values align 1:1 with words
words = datum.stimulus.unit_strings()
print("\n  Word -> Value (first 8):")
for w, v in list(zip(words, datum.values, strict=False))[:8]:
    print(
        f"    {w:15s} -> {v:7.1f} ms" if not np.isnan(v) else f"    {w:15s} ->     NaN"
    )

Choose the response measure explicitly. Here, `first_run_in_first_pass`
measures the first visit to a word when it was entered on the first pass.
Eye-tracking loaders exclude zero-duration skips by default. The
[segmentation and response guide](segmentation.ipynb) explains source measures,
skip policies and fixation recalculation.

## 2. Run an evaluation

- `EvaluationSpec` assigns features to targets, predictors and controls.
- `AnalysisConfig` sets the regression and cross-validation.
- `Design` declares grouping factors and random effects.

This example evaluates surprisal beyond word length using individual reading
times. Its default linear mixed-effects model (LMM) includes both participant
and item intercepts: an item is a word occurrence shared across readers, not a
word type. It also has independent participant surprisal slopes. Surprisal is
constant for a given word occurrence, so there is no within-occurrence predictor
variation from which to estimate a slope. A passage-level slope is a different, additional term
when surprisal varies within passages; see the [analysis guide](../guides/grouped-analysis.md).
It holds
out both readers and complete passages, targeting 90% of each in training
(approximately 81% of responses on a balanced reader–passage grid).
The default creates occurrence IDs shared by readers of the same source word.

In [ ]:
from replica import EvaluationSpec, ModelFeature, evaluate
from replica.feature import HuggingFaceLM, Surprisal, UnitLength

features = EvaluationSpec(
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength()],
    targets=[ds.feature("first_run_in_first_pass"), ds.feature("first_fixation")],
)

results = evaluate(ds, features)

These are the ordinary `AnalysisConfig()` defaults. The baseline removes
surprisal's fixed effect and random slope, and both models score the same
held-out observations. The comparison warning asks you to review that question;
`AnalysisConfig(design=Design(comparison="predictor"))` records that choice.
Occurrence IDs are created automatically. If stimuli are pages from a larger
source, declare that source grouping as explained in the
[analysis guide](../guides/grouped-analysis.md).

To change settings, pass an `AnalysisConfig`, for example
`AnalysisConfig(train_fraction=0.8)` or `AnalysisConfig(transform_dv=np.log)`.
The default uses up to 100 crossed folds per model and requires R with lme4.

`evaluate()` reports predictive scores. Call `predictive_test(ds, features)`
from `replica` for bootstrap significance inference with the same analysis
defaults. This requires substantially more fitting; the
[analysis guide](../guides/grouped-analysis.md) explains its procedure and budget.

## 3. Inspect the results

`evaluate()` returns one `AnalysisResult` per predictor comparison and response
measure. This example specifies two measures, so it returns two results.

In [ ]:
for r in results:
    print(f"{r.feature_name} -> {r.dv_name}")
    print(f"  delta-LL:          {r.delta_ll:+.2f}")
    print(f"  delta-LL per unit: {r.delta_ll_per_unit:+.6f}")
    print(f"  delta-R2:          {r.delta_r2:+.6f}")
    print(f"  n observations:    {r.n_observations}")
    print(f"  mean coefficient:  {r.coefficient:+.4f}")
    print()

- `delta_ll`: total held-out log-score gain over the baseline.
- `delta_ll_per_unit`: mean gain per scored response, in nats. Here one response
  is one participant's reading time for one word; it is not one unique word.
- `delta_r2`: dimensionless held-out R² gain, averaged across folds with each
  fold weighted by its number of scored responses. Within a fold,
  R² = 1 − squared prediction error / total squared deviation from the test mean.
  It is already normalized; dividing by the number of units again would make it
  depend arbitrarily on sample size.
- `coefficient`: the mean fixed regression coefficient across training fits.

A positive gain means the full model scored better in this evaluation; it is
not itself a significance claim. A positive surprisal coefficient means the
model predicts longer reading times as surprisal increases.

## 4. Find your next step

Change `HuggingFaceLM("gpt2")` to select another model, or edit the predictors,
controls and targets in `EvaluationSpec`. For a published reproduction, start
with its study recipe and README.

| Task | Guide |
|---|---|
| Compare several datasets or models | [Benchmarking](benchmarking.ipynb) |
| Explore dataset options | [Loading datasets](loading_any_dataset.ipynb) |
| Add a model, feature, dataset or study | [Extending REPLICA](extending_replica.ipynb) |
| Understand statistical choices and significance testing | [Analysis guide](../guides/grouped-analysis.md) |

The repository's main entry points are [dataset loaders](../../src/replica/dataset),
[features](../../src/replica/feature), [analysis](../../src/replica/analysis),
and [study recipes](../../src/replica/studies). To contribute, see the
[contributing instructions](../../README.md#contributing) and
[test guide](../../tests/README.md).